# Practice Lab: Structured Output & Tool Use

Module 5 · Lesson 5.5 · 4 exercises
**Needs:** google-generativeai, pydantic


# Lesson 5.5: Structured Output & Tool Use

4 exercises: 1E + 2M + 1C
**Needs:** `pip install google-generativeai pydantic`


In [ ]:
!pip install google-generativeai pydantic -q


In [ ]:
import google.generativeai as genai
import json, os, re, time
from pydantic import BaseModel, Field, ValidationError
from typing import Optional

genai.configure(
    api_key=os.getenv('GOOGLE_API_KEY'))


---
## Exercise 2 (Medium): Resume Parser
Nested Pydantic schemas for resume extraction.


In [ ]:
# YOUR CODE
class Experience(BaseModel):
    company: str
    role: str
    years: float = Field(ge=0, le=50)

class Education(BaseModel):
    degree: str
    institution: str
    year: int = Field(ge=1950, le=2030)

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    skills: list[str] = Field(min_length=1)
    experience: list[Experience] = Field(min_length=1)
    education: list[Education] = Field(min_length=1)

# TODO: build ValidatedOutput, parse 3 resumes


<details><summary>Solution</summary>


In [ ]:
class ValidatedOutput:
    def __init__(self, max_retries=3):
        self.max_retries = max_retries
        self.model = genai.GenerativeModel(
            'gemini-2.0-flash',
            generation_config={'temperature': 0.1})

    def _clean(self, text):
        text = re.sub(r'```json?\s*', '', text)
        text = text.replace('```', '').strip()
        s = text.find('{')
        e = text.rfind('}') + 1
        return text[s:e] if s >= 0 else text

    def generate(self, prompt, schema):
        full = (f'{prompt}\nReturn ONLY valid JSON '
                f'matching:\n'
                f'{json.dumps(schema.model_json_schema(), indent=2)}')
        err = None
        for i in range(self.max_retries):
            if err:
                full += f'\nError: {err}\nFix it.'
            resp = self.model.generate_content(
                full).text.strip()
            try:
                return schema.model_validate_json(
                    self._clean(resp))
            except Exception as e:
                err = str(e)[:200]
                print(f'  Retry {i+1}: {err[:60]}')
        raise ValueError('Failed')

validator = ValidatedOutput()

resumes = [
    'Priya Sharma, priya@gmail.com, +91 9876543210. '
    'Skills: Python, Django, PostgreSQL. '
    '3yr TCS Software Dev, 1.5yr Infosys Junior Dev. '
    'B.Tech IIT Hyderabad 2019.',
    'Arjun Patel, arjun.patel@outlook.com. '
    'Skills: Java, Spring Boot, AWS. '
    '5yr Wipro Tech Lead, 2yr Accenture Sr Dev. '
    'M.Tech NIT Warangal 2017, B.Tech JNTU 2015.',
    'Meera Reddy, meera.r@yahoo.com. '
    'Skills: React, Node.js, MongoDB. '
    '2yr Zoho Frontend Dev. '
    'BCA Osmania University 2022.',
]

for i, text in enumerate(resumes):
    print(f'\nResume {i+1}:')
    try:
        r = validator.generate(
            f'Extract resume: {text}', Resume)
        print(f'  {r.name} | {r.email}')
        print(f'  Skills: {", ".join(r.skills)}')
        for exp in r.experience:
            print(f'  - {exp.company} ({exp.role}, '
                  f'{exp.years}yr)')
    except Exception as e:
        print(f'  Error: {e}')


</details>


---
## Exercise 3 (Medium): Multi-Tool Chatbot
6 tools with function calling. EMI calc works locally.


In [ ]:
# EMI calculator - pure Python, verifiable
def calculate_emi(price, months=12, rate=12.0):
    """Calculate EMI for a product purchase."""
    r = rate / 12 / 100
    emi = price * r * (1+r)**months / (
        (1+r)**months - 1)
    return {'price': price, 'months': months,
            'rate': rate,
            'emi': round(emi, 2),
            'total': round(emi * months, 2)}

result = calculate_emi(79900, 12, 12.0)
print(f'EMI for Rs 79,900 @ 12% for 12 months:')
print(f'  Monthly: Rs {result["emi"]:,.2f}')
print(f'  Total:   Rs {result["total"]:,.2f}')
print(f'  Interest: Rs {result["total"] - 79900:,.2f}')


<details><summary>Solution (full chatbot)</summary>


In [ ]:
# Full 6-tool chatbot needs Gemini API
# See practice-lab-5_5.html for complete code
print('See HTML for full function calling code')


</details>


---
## Exercise 1 (Easy): JSON Extraction Race
See practice-lab-5_5.html for full code.


In [ ]:
# Exercise 1: JSON Extraction Race
# Needs Gemini API.
pass


---
## Exercise 4 (Challenge): Reliability Benchmark
See practice-lab-5_5.html for full code.


In [ ]:
# Exercise 4: Reliability Benchmark
# Needs Gemini API.
pass
